# Maximal Marginal Relevance (MMR) - Theoretical & Mathematical Formulation (Lecture 46)

This notebook implements the theoretical and mathematical foundations of **Maximal Marginal Relevance (MMR)** as introduced by Carbonell & Goldstein (1998) and presented in Lecture 46.

### The Redundancy Problem in Vector Retrieval
Standard dense retrievers rely on cosine similarity with the query vector. In corpora with overlapping documents, top-K search often returns near-identical passages, saturating the LLM context window with redundant tokens while starving it of complementary perspectives.

### The Optimization Formula
MMR greedily balances query relevance against inter-document redundancy at each selection step $t$:

$$\text{MMR} \triangleq \arg\max_{d_i \in R \setminus S} \left[ \lambda \cdot \text{Sim}_1(d_i, q) - (1 - \lambda) \cdot \max_{d_j \in S} \text{Sim}_2(d_i, d_j) \right]$$

Where:
- $q$: Query vector.
- $R$: Candidate documents pool ($|R| = \text{fetch\_k}$).
- $S$: Set of already selected documents.
- $R \setminus S$: Remaining unselected candidates.
- $\lambda \in [0, 1]$: Trade-off parameter weighting relevance vs. diversity.

## 1. Setup & Environment
Import scientific computing libraries and type definitions.

In [1]:
import numpy as np
from typing import List, Dict, Tuple, Any

print("Environment initialized.")

Environment initialized.


## 2. Programmatic Implementation of the Lecture 46 Worked Problem
Reproduce the exact numerical problem analyzed in Lecture 46.

In [2]:
# Query: 'How to use LangChain for RAG'
# D1: 'LangChain enables retrieval with FAISS'
# D2: 'LangChain can use Chroma and Pinecone too'
# D3: 'LangChain agents can call external tools'

sim_query: Dict[str, float] = {
    "D1": 0.95,
    "D2": 0.93,
    "D3": 0.80
}

# Pairwise document similarities (redundancy metric)
sim_pairwise: Dict[Tuple[str, str], float] = {
    ("D1", "D2"): 0.90,  # High redundancy
    ("D1", "D3"): 0.30,  # High diversity
    ("D2", "D3"): 0.40
}

lambda_param: float = 0.7

print(f"Similarity with Query: {sim_query}")
print(f"Pairwise Document Similarities: {sim_pairwise}")
print(f"Lambda Parameter: {lambda_param}")

Similarity with Query: {'D1': 0.95, 'D2': 0.93, 'D3': 0.8}
Pairwise Document Similarities: {('D1', 'D2'): 0.9, ('D1', 'D3'): 0.3, ('D2', 'D3'): 0.4}
Lambda Parameter: 0.7


## 3. Step 1: Greedy Selection of Top Relevant Document
Select initial document with maximum query cosine similarity.

In [3]:
# Step 1: Select document with highest query similarity
selected_docs: List[str] = [max(sim_query, key=sim_query.get)]
remaining_docs: List[str] = [doc for doc in sim_query if doc not in selected_docs]

print(f"Step 1: Selected initial document: {selected_docs[0]} (Sim: {sim_query[selected_docs[0]]})")
print(f"Remaining candidate pool: {remaining_docs}")

assert selected_docs[0] == "D1", "D1 must be selected first as it has highest query similarity" 

Step 1: Selected initial document: D1 (Sim: 0.95)
Remaining candidate pool: ['D2', 'D3']


## 4. Step 2: MMR Rescoring of Remaining Candidates
Compute MMR scores for unselected documents under lambda=0.7.

In [4]:
def get_pairwise_sim(doc_a: str, doc_b: str) -> float:
    return sim_pairwise.get((doc_a, doc_b)) or sim_pairwise.get((doc_b, doc_a), 0.0)

mmr_scores: Dict[str, float] = {}

for candidate in remaining_docs:
    relevance_term = lambda_param * sim_query[candidate]
    # Maximum similarity with any already selected document in S
    max_sim_to_selected = max(get_pairwise_sim(candidate, s_doc) for s_doc in selected_docs)
    redundancy_penalty = (1.0 - lambda_param) * max_sim_to_selected
    score = relevance_term - redundancy_penalty
    mmr_scores[candidate] = round(score, 4)
    print(f"MMR({candidate}) = ({lambda_param} * {sim_query[candidate]}) - ({1.0 - lambda_param:.1f} * {max_sim_to_selected})")
    print(f"         = {relevance_term:.4f} - {redundancy_penalty:.4f} = {mmr_scores[candidate]}\n")

assert mmr_scores["D2"] == 0.381, f"Expected 0.381 for D2, got {mmr_scores['D2']}"
assert mmr_scores["D3"] == 0.470, f"Expected 0.470 for D3, got {mmr_scores['D3']}"
print("Exact numerical assertion from Lecture 46 passed.")

MMR(D2) = (0.7 * 0.93) - (0.3 * 0.9)
         = 0.6510 - 0.2700 = 0.381

MMR(D3) = (0.7 * 0.8) - (0.3 * 0.3)
         = 0.5600 - 0.0900 = 0.47

Exact numerical assertion from Lecture 46 passed.


## 5. Step 3: Selection and Rank Assignment
Rank candidates based on computed MMR scores.

In [5]:
second_selected = max(mmr_scores, key=mmr_scores.get)
selected_docs.append(second_selected)

print("=== Final Ranked Document Selection ===")
print(f"Rank #1: {selected_docs[0]} (Reason: Highest Initial Query Relevance)")
print(f"Rank #2: {selected_docs[1]} (Reason: Maximum Marginal Relevance & Diversity)")

assert selected_docs == ["D1", "D3"], "Final selection must be D1 followed by D3"
print("\nVerification passed: D3 selected over D2 due to diversity penalty.")

=== Final Ranked Document Selection ===
Rank #1: D1 (Reason: Highest Initial Query Relevance)
Rank #2: D3 (Reason: Maximum Marginal Relevance & Diversity)

Verification passed: D3 selected over D2 due to diversity penalty.


## 6. General Purpose Vectorized MMR Implementation
A production-grade vectorized MMR implementation supporting arbitrary embedding matrices.

In [6]:
def maximal_marginal_relevance(
    query_vector: np.ndarray,
    candidate_vectors: np.ndarray,
    lambda_mult: float = 0.5,
    k: int = 2
) -> List[int]:
    """Compute MMR selection indices from normalized dense embedding vectors."""
    # Normalize vectors
    q_norm = query_vector / np.linalg.norm(query_vector)
    c_norm = candidate_vectors / np.linalg.norm(candidate_vectors, axis=1, keepdims=True)
    
    # Query similarities
    sim_to_query = np.dot(c_norm, q_norm)
    
    selected_indices: List[int] = [int(np.argmax(sim_to_query))]
    candidate_indices = [i for i in range(len(candidate_vectors)) if i not in selected_indices]
    
    while len(selected_indices) < min(k, len(candidate_vectors)):
        best_score = -float("inf")
        best_idx = None
        
        for cand_idx in candidate_indices:
            rel = lambda_mult * sim_to_query[cand_idx]
            div = (1.0 - lambda_mult) * np.max([
                np.dot(c_norm[cand_idx], c_norm[sel_idx])
                for sel_idx in selected_indices
            ])
            mmr_val = rel - div
            if mmr_val > best_score:
                best_score = mmr_val
                best_idx = cand_idx
                
        selected_indices.append(best_idx)
        candidate_indices.remove(best_idx)
        
    return selected_indices

# Test vectorized implementation
np.random.seed(42)
dim = 64
q_vec = np.random.randn(dim)
docs_vec = np.random.randn(10, dim)

res_indices = maximal_marginal_relevance(q_vec, docs_vec, lambda_mult=0.7, k=3)
print(f"Vectorized MMR selected indices: {res_indices}")
assert len(res_indices) == 3, "Must return exactly 3 indices"
print("Vectorized MMR assertion passed.")

Vectorized MMR selected indices: [4, 7, 6]
Vectorized MMR assertion passed.
